In [1]:
import os
import torch
import pickle
import numpy as np
import xarray as xr
from tqdm import tqdm
from data.image import img_to_task

/home/vinayakrana/miniconda3/envs/tnp/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/vinayakrana/miniconda3/envs/tnp/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


In [2]:
from tqdm import tqdm

In [3]:
val_ds = xr.open_dataset("datasets/WUSTL/scaled_val_data.nc")

In [4]:
val_ds

<xarray.Dataset>
Dimensions:  (time: 24, x1: 340, x2: 320)
Coordinates:
  * time     (time) datetime64[ns] 2009-01-01 2009-02-01 ... 2010-12-01
  * x1       (x1) float32 0.0 0.00295 0.0059 0.00885 ... 0.9941 0.9971 1.0
  * x2       (x2) float32 0.0 0.00295 0.0059 0.008849 ... 0.9351 0.9381 0.941
Data variables:
    PM25     (time, x1, x2) float32 ...
Attributes:
    TITLE:            Convolutional Neural Network Monthly PM2.5 Estimation o...
    CONTACT:          SIYUAN SHEN <s.siyuan@wustl.edu>
    LAT_DELTA:        0.1
    LON_DELTA:        0.1
    SPATIALCOVERAGE:  AS
    TIMECOVERAGE:     201206

In [15]:
val_ds.to_array().values[0][1].shape

(340, 320)

In [6]:
n_context_list = [5, 20, 50, 100, 200, 500]
seeds = [0, 1, 2, 3, 4]
progress_bar = tqdm(total=len(n_context_list) * len(seeds))
np.random.seed(0)
checksum = 0
for n_context in n_context_list:
    for seed in seeds:
        batches = []
        for timestamp in val_ds.time.values:
            img = torch.from_numpy(val_ds.sel(time=timestamp).to_array().values).unsqueeze(0).cuda()
            batch = img_to_task(img, seed=seed)
            batches.append(batch)
        save_path = f"evalsets/wustl/val_tasks/{n_context=}/{seed=}.pkl"
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        with open(save_path, "wb") as f:
            pickle.dump(batches, f)
        progress_bar.update(1)
        checksum += batches[0].xt.sum()
print(checksum)

100%|██████████| 30/30 [00:15<00:00,  1.27it/s]

tensor(1.0720e+09, device='cuda:0')


In [9]:
val_tasks = []
val_tasks_path = 'evalsets/wustl/val_tasks'
for context_folder in os.listdir(val_tasks_path):
    for task_file in os.listdir(f'{val_tasks_path}/{context_folder}'):
        
        with open(f'{val_tasks_path}/{context_folder}/{task_file}','rb') as f:
            val_tasks.extend(pickle.load(f))

In [23]:
for x,y in val_tasks[0].items():
    val_tasks[0][x] = y.cuda()

In [29]:
val_tasks[0].yt

tensor([[[-0.2971],
         [-0.3937],
         [-0.0355],
         ...,
         [ 0.7957],
         [-0.0280],
         [ 0.9839]]], device='cuda:0')